### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ljubljana_primary_tumor",
    dataset_year="1987", # from the cite date on UCI...
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5WK5Q",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/83/primary+tumor.zip && unzip primary+tumor.zip primary-tumor.data && rm primary+tumor.zip && mkdir -p local-data-warehouse/ljubljana_primary_tumor && mv primary-tumor.data local-data-warehouse/ljubljana_primary_tumor/
""",
    # References
    academic_reference_bibtex="""@misc{Zwitter1987primarytumor,
  author       = {Zwitter, M. and Soklic, M.},
  title        = {{Primary Tumor}},
  year         = {1987},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C5WK5Q}
}
""",
    academic_reference_bibtex_key="Zwitter1987primarytumor",
    licence="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- We encode missing values as np.nan instead of "?".
- We reverse the ordinal encoding of all features, adding the text back to the categories.
- We drop classes with less than 10 samples to have sufficient data per class for a robust benchmark task. In reality, this would call for collecting more data on these classes, or treating it as an individual few-shot prediction task.
- The data contains many duplicates (10%), which seems to be naturally occurring given the limited amount of features. Moreover, the duplicates sometimes have different targets. We keep the duplicates. While this might bias evaluation a bit, it represents the real-world task with ambiguous features better.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="class",
)

## Preprocessing

In [2]:
import pandas as pd

columns = [
    "class","age","sex","histologic-type","degree-of-diffe","bone","bone-marrow","lung","pleura","peritoneum","liver","brain","skin","neck","supraclavicular","axillar","mediastinum","abdominal"
]
df = pd.read_csv(dataset_mold.path / "primary-tumor.data", header=None, names=columns, na_values="?")
print("Loaded data shape:", df.shape)

domains = {
    "class": [
        "lung", "head & neck", "esophasus", "thyroid", "stomach",
        "duoden & sm.int", "colon", "rectum", "anus", "salivary glands",
        "pancreas", "gallblader", "liver", "kidney", "bladder", "testis",
        "prostate", "ovary", "corpus uteri", "cervix uteri", "vagina", "breast"
    ],
    "age": ["<30", "30-59", ">=60"],
    "sex": ["male", "female"],
    "histologic-type": ["epidermoid", "adeno", "anaplastic"],
    "degree-of-diffe": ["well", "fairly", "poorly"],
}

# yes/no attributes
yes_no_cols = [
    "bone", "bone-marrow", "lung", "pleura", "peritoneum", "liver",
    "brain", "skin", "neck", "supraclavicular", "axillar",
    "mediastinum", "abdominal"
]

for c in yes_no_cols:
    domains[c] = ["yes", "no"]

for col, vals in domains.items():
    df[col] = df[col].map({i+1: v for i, v in enumerate(vals)})

# We drop classes with less than 10 samples
df = df[df["class"].isin(df["class"].value_counts()[df["class"].value_counts() >= 10].index)]
as_cat_type = list(domains.keys())
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (339, 18)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 302
Columns: 18
Use sampling: False (sample size: 302)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['degree-of-diffe', 'histologic-type', 'age', 'skin', 'sex', 'axillar', 'bone', 'bone-marrow', 'lung', 'pleura']
Rows remaining as candidates after top-10 filter: 193 (of 302)

#### Duplicate Report
Total duplicate rows: 30 (9.93% of dataset)
Duplicate rows ignoring target: 49 (16.23% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,class,age,sex,histologic-type,degree-of-diffe,bone,bone-marrow,lung,pleura,peritoneum,liver,brain,skin,neck,supraclavicular,axillar,mediastinum,abdominal
0,pancreas,30-59,male,NaN,poorly,no,no,yes,no,yes,yes,no,no,no,no,no,yes,yes
1,kidney,30-59,male,NaN,NaN,yes,no,no,no,no,no,no,no,no,no,no,no,no
2,thyroid,30-59,female,adeno,poorly,yes,no,no,yes,no,yes,no,no,no,yes,no,yes,no
3,prostate,>=60,male,adeno,well,no,no,yes,no,no,yes,no,no,no,no,no,yes,yes
4,lung,30-59,female,anaplastic,poorly,no,no,no,yes,no,no,yes,no,no,no,no,yes,no


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,degree-of-diffe,category,136.0,45.03,3.0,"poorly, well, fairly"
1,histologic-type,category,61.0,20.20,3.0,"adeno, epidermoid, anaplastic"
2,sex,category,1.0,0.33,2.0,"female, male"
3,skin,category,1.0,0.33,2.0,"no, yes"
4,axillar,category,1.0,0.33,2.0,"no, yes"
5,class,category,0.0,0.00,11.0,"lung, stomach, ovary, pancreas, kidney, breast, head & neck, gallblader, colon, thyroid"
6,age,category,0.0,0.00,3.0,"30-59, >=60, <30"
7,bone,category,0.0,0.00,2.0,"no, yes"
8,bone-marrow,category,0.0,0.00,2.0,"no, yes"
9,lung,category,0.0,0.00,2.0,"no, yes"


In [6]:
# Numeric Feature Statistics
numeric_stats

'No numeric features to summarize.'

In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column          rank                          
abdominal       1             no    203  67.22
                2            yes     99  32.78
age             1          30-59    186  61.59
                2           >=60     94  31.13
                3            <30     22   7.28
axillar         1             no    269  89.07
                2            yes     32  10.60
                3           <NA>      1   0.33
bone            1             no    218  72.19
                2            yes     84  27.81
bone-marrow     1             no    295  97.68
                2            yes      7   2.32
brain           1             no    283  93.71
                2            yes     19   6.29
class           1           lung     84  27.81
                2        stomach     39  12.91
                3          ovary     29   9.60
                4       pancreas     28   9.27
                5         kidney     24   7.95
degree-of-diffe 1           <NA>    136  45.03
                2         poorly     92  30.46
                3           well     51  16.89
                4         fairly     23   7.62
histologic-type 1          adeno    197  65.23
                2           <NA>     61  20.20
                3     epidermoid     37  12.25
                4     anaplastic      7   2.32
liver           1             no    203  67.22
                2            yes     99  32.78
lung            1             no    240  79.47
                2            yes     62  20.53
mediastinum     1             no    217  71.85
                2            yes     85  28.15
neck            1             no    265  87.75
                2            yes     37  12.25
peritoneum      1             no    216  71.52
                2            yes     86  28.48
pleura          1             no    235  77.81
                2            yes     67  22.19
sex             1         female    159  52.65
                2           male    142  47.02
                3           <NA>      1   0.33
skin            1             no    283  93.71
                2            yes     18   5.96
                3           <NA>      1   0.33
supraclavicular 1             no    248  82.12
                2            yes     54  17.88

In [8]:
# Target Distribution
target_df

,count,pct
class,,
lung,84,27.81
stomach,39,12.91
ovary,29,9.60
pancreas,28,9.27
kidney,24,7.95
breast,24,7.95
head & neck,20,6.62
gallblader,16,5.30
colon,14,4.64


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c80b5-f3a2-7254-890f-c728e070cfed
3aef2b8c96e99c0d291c35a9cf0361cb126bf30083a64bace2d84dd1103bf667
